# Java Source Code OOP Metrics Analysis
This notebook analyzes the `src` folder of the PlatformerGame project and writes a structured metrics file.

## 1. Import Required Libraries
Import Python libraries for file I/O, regular expressions, and JSON output.

In [ ]:
import pathlib
import re
import json
from collections import Counter

## 2. Load Source Code Files
Read all Java source files from `src` into memory for analysis.

In [ ]:
workspace_root = pathlib.Path(r'c:/Users/Giovanni/OneDrive/Documents/GitHub/PlatformerGame#')
src_root = workspace_root / 'src'
java_files = sorted(src_root.rglob('*.java'))
len(java_files)

## 3. Parse Java Source Code
Use regular expressions to identify classes, methods, constructors, and type uses.

In [ ]:
class_pattern = re.compile(r'^(?:\s*(public|protected|private)?\s*)?(abstract\s+)?(class|interface|enum)\s+([A-Za-z_][A-Za-z0-9_]*)', re.MULTILINE)
method_pattern = re.compile(r'^(?P<prefix>\s*(?:public|protected|private|static|final|synchronized|abstract|native|transient|volatile|\s)+)?\s*(?P<type>[\w<>\[\]]+)\s+(?P<name>[A-Za-z_][A-Za-z0-9_]*)\s*\([^;{]*\)\s*(?:throws\s+[^{]+)?\s*\{', re.MULTILINE)
constructor_pattern = re.compile(r'^(?P<prefix>\s*(?:public|protected|private)?\s*)?(?P<name>[A-Za-z_][A-Za-z0-9_]*)\s*\([^;{]*\)\s*\{', re.MULTILINE)
field_pattern = re.compile(r'^(?P<prefix>\s*(?:public|protected|private|static|final|transient|volatile|synchronized|\s)+)?\s*(?P<type>[A-Za-z0-9_<>,\[\].]+)\s+(?P<name>[A-Za-z_][A-Za-z0-9_]*)\s*(?:=|;|,)', re.MULTILINE)
def normalize_type(t):
    t = t.strip()
    t = re.sub(r'\s+', ' ', t)
    t = re.sub(r'<.*>', lambda m: '<' + re.sub(r'\s+', ' ', m.group(0)[1:-1]) + '>', t)
    return t
text_by_file = {}
for file_path in java_files:
    text = file_path.read_text(encoding='utf-8')
    text_by_file[file_path] = re.sub(r'//.*|/\*.*?\*/', '', text, flags=re.S)

## 4. Count Try/Catch Blocks
Count `try`, `catch`, and `finally` blocks in the cleaned source text.

In [ ]:
try_count = 0
catch_count = 0
finally_count = 0
for text in text_by_file.values():
    try_count += len(re.findall(r'\btry\b', text))
    catch_count += len(re.findall(r'\bcatch\b', text))
    finally_count += len(re.findall(r'\bfinally\b', text))
try_count, catch_count, finally_count

## 5. Count Methods, Constructors, Setters, and Getters
Detect method declarations, constructors, and common setter/getter naming patterns.

In [ ]:
metrics = {
    'classes': 0,
    'abstract_classes': 0,
    'interfaces': 0,
    'enums': 0,
    'methods': 0,
    'constructors': 0,
    'getters': 0,
    'setters': 0,
    'fields': 0,
    'switch_statements': 0,
    'case_labels': 0,
    'public_modifiers': 0,
    'private_modifiers': 0,
    'protected_modifiers': 0,
    'types': Counter(),
}
for file_path, text in text_by_file.items():
    class_names = [m.group(4) for m in class_pattern.finditer(text)]
    for m in class_pattern.finditer(text):
        kind = m.group(3)
        abstract_flag = bool(m.group(2))
        if kind == 'class':
            if abstract_flag:
                metrics['abstract_classes'] += 1
            else:
                metrics['classes'] += 1
        elif kind == 'interface':
            metrics['interfaces'] += 1
        elif kind == 'enum':
            metrics['enums'] += 1
    for m in method_pattern.finditer(text):
        name = m.group('name')
        if name in class_names:
            continue
        metrics['methods'] += 1
        if name.startswith('get') or name.startswith('is'):
            metrics['getters'] += 1
        if name.startswith('set'):
            metrics['setters'] += 1
        metrics['types'][normalize_type(m.group('type'))] += 1
    for m in constructor_pattern.finditer(text):
        if m.group('name') in class_names:
            metrics['constructors'] += 1
    for m in field_pattern.finditer(text):
        metrics['fields'] += 1
        metrics['types'][normalize_type(m.group('type'))] += 1
    metrics['public_modifiers'] += len(re.findall(r'\bpublic\b', text))
    metrics['private_modifiers'] += len(re.findall(r'\bprivate\b', text))
    metrics['protected_modifiers'] += len(re.findall(r'\bprotected\b', text))
    metrics['switch_statements'] += len(re.findall(r'\bswitch\b', text))
    metrics['case_labels'] += len(re.findall(r'\bcase\b', text))
metrics

## 6. Analyze Access Modifiers and Data Types
Summarize the modifiers and all detected data type tokens used in declarations.

In [ ]:
type_counts = sorted(metrics['types'].items(), key=lambda x: (-x[1], x[0]))
type_counts[:40]

## 7. Count Switch Cases
Report switch statement and case label counts across the codebase.

In [ ]:
switch_count = metrics['switch_statements']
case_count = metrics['case_labels']
switch_count, case_count

## 8. Save Structured Results to File
Write the collected metrics to a JSON and TXT file in the repository root.

In [ ]:
summary = {
    'java_files': len(java_files),
    'classes': metrics['classes'],
    'abstract_classes': metrics['abstract_classes'],
    'interfaces': metrics['interfaces'],
    'enums': metrics['enums'],
    'methods': metrics['methods'],
    'constructors': metrics['constructors'],
    'getters': metrics['getters'],
    'setters': metrics['setters'],
    'try_blocks': try_count,
    'catch_blocks': catch_count,
    'finally_blocks': finally_count,
    'switch_statements': switch_count,
    'case_labels': case_count,
    'access_modifiers': {
        'public': metrics['public_modifiers'],
        'private': metrics['private_modifiers'],
        'protected': metrics['protected_modifiers'],
    },
    'data_types': dict(type_counts),
}
json_path = workspace_root / 'project_oop_metrics.json'
txt_path = workspace_root / 'project_oop_metrics.txt'
json_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
with txt_path.open('w', encoding='utf-8') as f:
    f.write('PlatformerGame OOP Metrics\n')
    f.write('=======================\n')
    for key, value in summary.items():
        if key not in ('data_types', 'access_modifiers'):
            f.write(f'{key}: {value}\n')
    f.write('access_modifiers:\n')
    for mod, count in summary['access_modifiers'].items():
        f.write(f'  {mod}: {count}\n')
    f.write('data_types:\n')
    for dtype, count in summary['data_types'].items():
        f.write(f'  {dtype}: {count}\n')
json_path, txt_path